# ATAG Waypoint 2050 (3rd edition, light) - Scenario S0

The 3rd edition has no S0, so this reference scenario is built from the `_a_r_m`
publication notebook, which models a single **generic SAF** under regional (RefuelEU /
US) blending policies. The `_a_r_m` notebook runs a 20-region **multi-regional** process;
here we run a single **global** process instead.

## Where each input comes from

| Input group | Source |
| --- | --- |
| **Traffic** | 3rd edition full **S1** (central traffic growth) - `s1_inputs.json` |
| **Efficiency** | 3rd edition full **T2** technology variant (energy-per-ASK gains) - `t2_inputs.json` |
| **Operations** | operations final gain = **3** (kept from S1; S1/S2 light use 6) |
| **SAF** | **aggregated** generic-SAF fuel policy across all 20 `_a_r_m` regions (global effective share + energy-weighted emission factor) |
| **Offsets** | 3rd edition full **S1** carbon-offset parameters |

The SAF aggregation and the multi-region -> single global conversion are done by
`aeromaps.utils.single_region.aggregate_regions_to_single_process`.

## Build the S0 demand inputs

Traffic + operations (final gain 3) + offsets come from 3rd-edition S1; the aircraft
**efficiency** (energy-per-ASK gains) is overridden with the 3rd-edition **T2** values.

In [ ]:
# The scenario ships with the package; copy it somewhere writable before running,
# so this notebook's outputs and regenerated inputs land in ./workdir rather than
# in the installed AeroMAPS.
from aeromaps.utils.scenarios import prepare_scenario
import os
import json

SCENARIO = prepare_scenario("atag_3rd_edition_light")

FULL = "../3rd_edition_full/data_inputs"

with open(f"{FULL}/s1_inputs.json") as f:
    s0_inputs = json.load(f)  # traffic, load factor, offsets
with open(f"{FULL}/t2_inputs.json") as f:
    t2_inputs = json.load(f)  # technology variant 2

# Efficiency: adopt T2 energy-per-ASK gain trajectories, per passenger market.
# (Market-prefixed names since main's generic markets refactor.)
EFFICIENCY_KEYS = [
    f"{mid}_energy_per_ask_dropin_fuel_gain_reference_years{suffix}"
    for mid in ("short_range", "medium_range", "long_range")
    for suffix in ("", "_values")
]
for key in EFFICIENCY_KEYS:
    s0_inputs[key] = t2_inputs[key]

# Operations: S0 keeps a final gain of 3 (S1/S2 use 6), so set it explicitly
# rather than inheriting S1's value.
s0_inputs["operations_gain_reference_years"] = [2020, 2050]
s0_inputs["operations_gain_reference_years_values"] = [0, 3]

# Offsets: the post-CORSIA schedule is scenario-specific, because it is derived
# from each scenario's own gross trajectory by make_offset_glide.py. S0's gross
# is far higher than S1's, so inheriting S1's schedule here would leave S0's net
# line rising after the handover. Keep whatever is already in s0_inputs.json.
OFFSET_KEYS = [
    "residual_carbon_offset_share_reference_years",
    "residual_carbon_offset_share_reference_years_values",
]
if os.path.exists(str(SCENARIO / "data_inputs" / "s0_inputs.json")):
    with open(str(SCENARIO / "data_inputs" / "s0_inputs.json")) as f:
        _previous = json.load(f)
    for key in OFFSET_KEYS:
        if key in _previous:
            s0_inputs[key] = _previous[key]

with open(str(SCENARIO / "data_inputs" / "s0_inputs.json"), "w") as f:
    json.dump(s0_inputs, f, indent=4)
print(
    "Wrote data_inputs/s0_inputs.json (traffic+offsets from S1, efficiency from T2, operations gain 3)"
)

## Build the global process with an aggregated SAF policy

Every `_a_r_m` region is computed once; their heterogeneous SAF policies (EU 70% share,
US quantity mandate, 12 regions with no SAF, ...) are aggregated into a single global
effective SAF share and an energy-weighted emission factor, then run globally on the S0
demand and the top-down model chain (`config_s0_base.yaml`). This computes 20 regional
processes, so it takes a little longer than S1/S2.

In [ ]:
%matplotlib widget
from aeromaps.utils.single_region import aggregate_regions_to_single_process

# The _a_r_m regions pin no historic baseline of their own, so they would run on
# whatever the packaged parameters.json carries -- 2000-2019 observed, prospection
# from 2020. This scenario is observed through 2023 and must be comparable with the
# full edition's S1 and S2, so it hands the regions its own baseline rather than
# moving the shipped default, which every other scenario in the repository reads.
BASELINE_KEYS = (
    "historic_start_year",
    "prospection_start_year",
    "rpk_init",
    "ask_init",
    "rtk_init",
    "pax_init",
    "freight_init",
    "energy_consumption_init",
    "total_aircraft_distance_init",
)
with open(str(SCENARIO / "data_inputs" / "s0_inputs.json")) as f:
    _s0_inputs = json.load(f)
region_baseline = {k: _s0_inputs[k] for k in BASELINE_KEYS}
print(
    f"regions rebaselined to {region_baseline['historic_start_year']}-"
    f"{region_baseline['prospection_start_year'] - 1} observed"
)

process = aggregate_regions_to_single_process(
    configuration_file="../../../publications/_a_r_m/regionalisation_all_regions.yaml",
    region_baseline=region_baseline,
    output_config=str(SCENARIO / "config_files" / "config_s0.yaml"),
    output_energy_file=str(SCENARIO / "data_inputs" / "s0_energy.yaml"),
    demand_override=str(SCENARIO / "data_inputs" / "s0_inputs.json"),
    standards_override=str(SCENARIO / "config_files" / "config_s0_base.yaml"),
    fuel_carrier="generic_saf",
    reference_region="EU_DOM",
    output_json="data_outputs/s0.json",
)
print("Process type:", type(process).__name__)

## Compute

In [ ]:
process.compute()
process.write_json()
print("Global aggregated SAF share (%):")
print(process.data["vector_outputs"]["generic_saf_mandate_share"].loc[[2025, 2030, 2040, 2050]])

## Results

In [ ]:
process.plot("air_transport_co2_emissions")

In [ ]:
process.plot("fuel_shares")

In [ ]:
process.plot("energy_mix")

### Where the fuel price comes from

Per-pathway minimum fuel selling price, split into the pathway's own cost and each
resource it consumes. Note that green electricity and DAC-CO2 carry a zero price here:
the report-derived pathway costs already include them, so pricing the resources again
would double-count. Green electricity likewise carries a zero emission factor, since
the report's carbon-intensity table is already a life-cycle figure. See the note in
`data_inputs/resources.yaml`.

In [ ]:
process.plot("mfsp_detailled")

### The same scenario, tank-to-wake

The reports headline tank-to-wake emissions, which count combustion only and, under
the CORSIA accounting they follow, credit a SAF pathway's upstream reduction as a
tank-to-wake reduction. This reproduction runs well-to-wake, so the two scopes are
not comparable without stating both. The twin below reads the same scenario against
tank-to-wake factors derived from the well-to-wake ones by `aeromaps.utils.emission_scopes`,
and writes `data_outputs/s0-TTW.json` beside the well-to-wake result.

In [ ]:
# The tank-to-wake twin is derived from the well-to-wake file rather than
# maintained as a second copy, so regenerate it before reading it. The
# conversion lives in aeromaps.utils.emission_scopes; it can also be applied
# in place to a built process, which is what apply_corsia_scope is for.
from aeromaps import create_process
from aeromaps.utils.emission_scopes import lifecycle_to_corsia

lifecycle_to_corsia(
    str(SCENARIO / "data_inputs" / "s0_energy.yaml"),
    str(SCENARIO / "data_inputs" / "s0-TTW_energy.yaml"),
)

process_ttw = create_process(
    configuration_file=str(SCENARIO / "config_files" / "config_s0-TTW.yaml")
)
process_ttw.compute()
process_ttw.write_json()
process_ttw.plot("air_transport_co2_emissions")